In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from pathlib import Path
import numpy as np

Mounted at /content/drive


In [ ]:
#Some helper functions

def remove_accents(text): #remove accents from greek text
    nfd = unicodedata.normalize('NFD', text)
    return ''.join(char for char in nfd if unicodedata.category(char) != 'Mn')

def load_masked_model(model_name):
    """Load any masked LM"""
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.eval()
    print("Model loaded!")
    return tokenizer, model

In [ ]:
def load_phenomenon_pairs(phenomenon_path):
    """Load pairs for one phenomenon"""
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"

    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()

    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()

    pairs = list(zip(gram_sentences, ungram_sentences))
    return pairs

def load_all_phenomena(data_dir):
    """Load all phenomena"""
    all_pairs = {}
    data_path = Path(data_dir)

    for phenomenon_folder in data_path.iterdir():
        if phenomenon_folder.is_dir():
            phenomenon_name = phenomenon_folder.name

            try:
                pairs = load_phenomenon_pairs(phenomenon_folder)
                all_pairs[phenomenon_name] = pairs
                print(f"✓ Loaded {phenomenon_name}: {len(pairs)} pairs")
            except FileNotFoundError:
                print(f"✗ Skipped {phenomenon_name}: files not found")

    return all_pairs

In [ ]:
# test loading all phenomena
all_data = load_all_phenomena("/content/drive/MyDrive/Thesis/data/phenomena")

print(f"\nTotal phenomena loaded: {len(all_data)}")
print("\nSummary:")
for phen_name, pairs in all_data.items():
    print(f"  - {phen_name}: {len(pairs)} pairs")

✓ Loaded case_selection: 30 pairs
✓ Loaded negations: 30 pairs
✗ Skipped clitics: files not found
✓ Loaded noun_adjective_agreement: 30 pairs
✓ Loaded einai_agreement: 30 pairs
✗ Skipped aspect: files not found
✗ Skipped subject_verb_agreement: files not found

Total phenomena loaded: 4

Summary:
  - case_selection: 30 pairs
  - negations: 30 pairs
  - noun_adjective_agreement: 30 pairs
  - einai_agreement: 30 pairs


In [ ]:
class MaskedLMScorer:

    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def score_sentence(self, sentence):
        """Score a sentence using PLL"""
        sentence = remove_accents(sentence.lower())
        input_ids = self.tokenizer.encode(sentence, add_special_tokens=True)
        input_tensor = torch.tensor([input_ids])

        score = 0
        num_scored_tokens = 0

        with torch.no_grad():
            for i in range(1, len(input_ids) - 1):
                masked_input = input_tensor.clone()
                masked_input[0, i] = self.tokenizer.mask_token_id

                outputs = self.model(masked_input)
                predictions = outputs.logits

                probs = torch.nn.functional.log_softmax(predictions[0, i], dim=-1)
                token_prob = probs[input_ids[i]].item()
                score += token_prob
                num_scored_tokens += 1

        #Return average
        avg_log_prob = score / num_scored_tokens if num_scored_tokens > 0 else 0
        avg_surprisal = -avg_log_prob

        return {
            'avg_log_prob': avg_log_prob,  #length-normalized
            'avg_surprisal': avg_surprisal,
            'num_tokens': num_scored_tokens
        }

In [ ]:
"""
# Load Greek BERT
model_name = "nlpaueb/bert-base-greek-uncased-v1"
tokenizer, model = load_masked_model(model_name)
"""

In [ ]:
"""
# Load mBERT
model_name = "google-bert/bert-base-multilingual-cased"
tokenizer, model = load_masked_model(model_name)
"""

In [ ]:
# Load XLM Roberta
model_name = "FacebookAI/xlm-roberta-base"
tokenizer, model = load_masked_model(model_name)


Loading FacebookAI/xlm-roberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


In [ ]:
# Create scorer
scorer = MaskedLMScorer(tokenizer, model)

# Test with minimal pair
ungram = "Το παιδιά τρώει"  # ungrammatical
gram = "Το παιδί τρώει"     # grammatical

result1 = scorer.score_sentence(ungram)
result2 = scorer.score_sentence(gram)

print(f"Ungrammatical: '{ungram}'")
print(f"  PLL: {result1['avg_log_prob']:.2f}")
print(f"  Surprisal: {result1['avg_surprisal']:.2f}")

print(f"\nGrammatical: '{gram}'")
print(f"  PLL: {result2['avg_log_prob']:.2f}")
print(f"  Surprisal: {result2['avg_surprisal']:.2f}")

print(f"\nModel prefers grammatical: {result2['avg_log_prob'] > result1['avg_log_prob']}")

Ungrammatical: 'Το παιδιά τρώει'
  PLL: -1.77
  Surprisal: 1.77

Grammatical: 'Το παιδί τρώει'
  PLL: -0.81
  Surprisal: 0.81

Model prefers grammatical: True


In [ ]:
import random

def sample_size_stability(scorer, all_data, sample_sizes=[25, 50, 75, 100], n_repeats=5):
    """
    For each phenomenon, score every pair ONCE, then resample at different
    sizes to see how accuracy estimate changes/stabilizes with more pairs.
    """
    results = {}

    for phenomenon, pairs in all_data.items():
        print(f"\nPhenomenon: {phenomenon} ({len(pairs)} pairs available)")

        # Score every pair once — reuse for all sample sizes (avoids rescoring)
        pair_correctness = []
        for gram, ungram in pairs:
            g = scorer.score_sentence(gram)
            u = scorer.score_sentence(ungram)
            pair_correctness.append(g['avg_log_prob'] > u['avg_log_prob'])

        phen_results = {}
        for size in sample_sizes:
            if size > len(pair_correctness):
                print(f"  ⚠ Skipping n={size}: only {len(pair_correctness)} pairs available")
                continue

            accs = []
            for rep in range(n_repeats):
                random.seed(rep)  # different, but reproducible, sample each repeat
                sample = random.sample(pair_correctness, size)
                accs.append(sum(sample) / size)

            phen_results[size] = {
                'mean_accuracy': sum(accs) / len(accs),
                'min_accuracy': min(accs),
                'max_accuracy': max(accs),
                'all_runs': accs
            }
            print(f"  n={size}: mean={phen_results[size]['mean_accuracy']:.2%} "
                  f"(range: {phen_results[size]['min_accuracy']:.2%}-{phen_results[size]['max_accuracy']:.2%})")

        results[phenomenon] = phen_results

    return results


# Run it
print("="*60)
print("SAMPLE-SIZE STABILITY EXPERIMENT")
print("="*60)
stability_results = sample_size_stability(scorer, all_data)

In [ ]:
# Store detailed results for all phenomena
all_results = {}

for phenomenon_name, pairs in all_data.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {phenomenon_name}")
    print(f"{'='*50}")

    correct_count = 0
    total_count = len(pairs)

    # Store detailed pair results
    pair_results = []

    for gram, ungram in pairs:
        # Score both
        gram_result = scorer.score_sentence(gram)
        ungram_result = scorer.score_sentence(ungram)

        # Check if correct (grammatical has higher PLL = lower surprisal)
        is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']

        if is_correct:
            correct_count += 1

        # Store detailed results for this pair
        pair_results.append({
            'grammatical': gram.strip(),
            'ungrammatical': ungram.strip(),
            'gram_pll': gram_result['avg_log_prob'],
            'ungram_pll': ungram_result['avg_log_prob'],
            'gram_surprisal': gram_result['avg_surprisal'],
            'ungram_surprisal': ungram_result['avg_surprisal'],
            'correct': is_correct
        })

    # Calculate accuracy and average metrics
    accuracy = correct_count / total_count
    avg_gram_pll = sum(p['gram_pll'] for p in pair_results) / len(pair_results)
    avg_ungram_pll = sum(p['ungram_pll'] for p in pair_results) / len(pair_results)
    avg_gram_surprisal = sum(p['gram_surprisal'] for p in pair_results) / len(pair_results)
    avg_ungram_surprisal = sum(p['ungram_surprisal'] for p in pair_results) / len(pair_results)

    # Store results
    all_results[phenomenon_name] = {
        'correct': correct_count,
        'total': total_count,
        'accuracy': accuracy,
        'avg_gram_pll': avg_gram_pll,
        'avg_ungram_pll': avg_ungram_pll,
        'avg_gram_surprisal': avg_gram_surprisal,
        'avg_ungram_surprisal': avg_ungram_surprisal,
        'pairs': pair_results  # All detailed pair data
    }

    print(f"Accuracy: {correct_count}/{total_count} = {accuracy:.2%}")
    print(f"Avg grammatical PLL: {avg_gram_pll:.2f}")
    print(f"Avg ungrammatical PLL: {avg_ungram_pll:.2f}")
    print(f"Avg grammatical surprisal: {avg_gram_surprisal:.2f}")
    print(f"Avg ungrammatical surprisal: {avg_ungram_surprisal:.2f}")

# Summary table
print(f"\n{'='*70}")
print(f"SUMMARY - Greek BERT")
print(f"{'='*70}")
print(f"{'Phenomenon':<30} {'Accuracy':<12} {'Gram PLL':<12} {'Ungram PLL':<12}")
print(f"{'-'*70}")
for phen, result in all_results.items():
    print(f"{phen:<30} {result['accuracy']:>10.2%} {result['avg_gram_pll']:>11.2f} {result['avg_ungram_pll']:>11.2f}")

# Overall
total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs
print(f"{'-'*70}")
print(f"{'Overall':<30} {overall_acc:>10.2%} ({total_correct}/{total_pairs})")
print(f"{'='*70}")


Evaluating: case_selection
Accuracy: 26/30 = 86.67%
Avg grammatical PLL: -2.35
Avg ungrammatical PLL: -2.84
Avg grammatical surprisal: 2.35
Avg ungrammatical surprisal: 2.84

Evaluating: negations
Accuracy: 23/30 = 76.67%
Avg grammatical PLL: -1.66
Avg ungrammatical PLL: -1.87
Avg grammatical surprisal: 1.66
Avg ungrammatical surprisal: 1.87

Evaluating: noun_adjective_agreement
Accuracy: 26/30 = 86.67%
Avg grammatical PLL: -2.22
Avg ungrammatical PLL: -2.88
Avg grammatical surprisal: 2.22
Avg ungrammatical surprisal: 2.88

Evaluating: einai_agreement
Accuracy: 16/30 = 53.33%
Avg grammatical PLL: -2.22
Avg ungrammatical PLL: -2.22
Avg grammatical surprisal: 2.22
Avg ungrammatical surprisal: 2.22

SUMMARY - Greek BERT
Phenomenon                     Accuracy     Gram PLL     Ungram PLL  
----------------------------------------------------------------------
case_selection                     86.67%       -2.35       -2.84
negations                          76.67%       -1.66       -1.87

In [ ]:
# Evaluate with ALL metrics tracked
all_results = {}

for phenomenon_name, pairs in all_data.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {phenomenon_name}")
    print(f"{'='*50}")

    correct_count = 0
    total_count = len(pairs)
    pair_results = []

    for gram, ungram in pairs:
        gram_result = model_scorer.score_sentence(gram)
        ungram_result = model_scorer.score_sentence(ungram)

        is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']
        if is_correct:
            correct_count += 1

        pair_results.append({
            'grammatical': gram.strip(),
            'ungrammatical': ungram.strip(),
            'gram_avg_log_prob': gram_result['avg_log_prob'],
            'ungram_avg_log_prob': ungram_result['avg_log_prob'],
            'gram_surprisal': gram_result['avg_surprisal'],
            'ungram_surprisal': ungram_result['avg_surprisal'],
            'gram_num_tokens': gram_result['num_tokens'],
            'ungram_num_tokens': ungram_result['num_tokens'],
            'correct': is_correct
        })

    accuracy = correct_count / total_count
    avg_gram_log_prob = sum(p['gram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_ungram_log_prob = sum(p['ungram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_gram_surprisal = sum(p['gram_surprisal'] for p in pair_results) / len(pair_results)
    avg_ungram_surprisal = sum(p['ungram_surprisal'] for p in pair_results) / len(pair_results)

    all_results[phenomenon_name] = {
        'correct': correct_count,
        'total': total_count,
        'accuracy': accuracy,
        'avg_gram_log_prob': avg_gram_log_prob,
        'avg_ungram_log_prob': avg_ungram_log_prob,
        'avg_gram_surprisal': avg_gram_surprisal,
        'avg_ungram_surprisal': avg_ungram_surprisal,
        'pairs': pair_results
    }

    print(f"Accuracy: {correct_count}/{total_count} = {accuracy:.2%}")
    print(f"Avg grammatical log prob: {avg_gram_log_prob:.2f}")
    print(f"Avg ungrammatical log prob: {avg_ungram_log_prob:.2f}")
    print(f"Avg grammatical surprisal: {avg_gram_surprisal:.2f}")
    print(f"Avg ungrammatical surprisal: {avg_ungram_surprisal:.2f}")

# Full summary table
print(f"\n{'='*90}")
print(f"COMPLETE SUMMARY")
print(f"{'='*90}")
print(f"{'Phenomenon':<25} {'Accuracy':<12} {'Gram LogP':<12} {'Ungram LogP':<12} {'Gram Surp':<12}")
print(f"{'-'*90}")
for phen, result in all_results.items():
    print(f"{phen:<25} {result['accuracy']:>10.2%} "
          f"{result['avg_gram_log_prob']:>11.2f} "
          f"{result['avg_ungram_log_prob']:>11.2f} "
          f"{result['avg_gram_surprisal']:>11.2f}")

total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs
print(f"{'-'*90}")
print(f"{'Overall':<25} {overall_acc:>10.2%} ({total_correct}/{total_pairs})")

Results saved to /content/drive/MyDrive/Thesis/results/masked/xlm_roberta_results.json
